# **TFM Project - Machine Learning for Drug Discovery in Neurodegenerative Diseases**
# **[Part 2] Pre-processed data**

Carla D. Di Monno

In **Part 2**, I focused on recovering missing bioactivity values, resolving duplicates, and ensuring high-quality labeling for machine learning models.

---

## **Installing libraries**

In [ ]:
!pip install rdkit

## **Importing libraries**

In [ ]:
import pandas as pd
import numpy as np
import io
from google.colab import files
from rdkit import Chem

In [ ]:
# Upload files
def upload_files (index_fields=None):
  uploaded = files.upload()
  for fn in uploaded.keys():
    print('User uploaded file "{name}" with length {length} bytes'.format(
        name=fn, length=len(uploaded[fn])))
    df = pd.read_csv(io.StringIO(uploaded[fn].decode('utf-8')), index_col = index_fields)
    return df, fn

## **Systematizing the data processing for multiple targets**

To make the entire workflow reusable for different therapeutic targets, I define a function that encapsulates all the functions involved in the data cleaning, curation, and labeling steps.

This function, `process_bioactivity_target`, take the raw DataFrame and the target's name as input. It will then apply all transformations, save the preprocessed and curated datasets, and return the final DataFrame.

### **1- Handling missing data**

**If any compounds has missing value for the *standard_value* and *canonical_smiles* column, the row will be deleted.**

In [ ]:
def handle_initial_missing_data(raw_df):
    df_cleaned = raw_df.dropna(subset=['standard_value', 'canonical_smiles']).copy()
    print(f"Records after dropping nulls in 'standard_value'/'canonical_smiles': {len(df_cleaned)}")
    return df_cleaned

### **2- Recovering missing bioactivity values**

In [ ]:
def rescue_pchembl(row):
    if pd.notnull(row['pchembl_value']):
      return row['pchembl_value']

    try:
        # Only proceed if standard_relation is exactly '='
        if row['standard_relation'] == '=':
            value = float(row['standard_value'])
            unity = str(row['standard_units'])

            if value <= 0: return None

            # pChEMBL = -log10(M)
            if unity == 'nM':
                return 9 - np.log10(value)
            elif unity in ['uM', 'µM', 'micromolar']:
                return 6 - np.log10(value)
        else:
            return None # Return None if standard_relation is not '='
    except:
        return None
    return None

def process_pchembl_values(df):
    pChEMBL_nulls = df['pchembl_value'].isnull().sum()
    print(f"Records without pChEMBL: {pChEMBL_nulls} ({pChEMBL_nulls/len(df)*100:.2f}%)")

    df.loc[:, 'pchembl_value'] = df.apply(rescue_pchembl, axis=1)
    df_cleaned_pchembl = df.dropna(subset=['pchembl_value']).copy()
    print(f"Records after rescuing and dropping null 'pchembl_value': {len(df_cleaned_pchembl)}")
    return df_cleaned_pchembl

### **3- Handling duplicated data**

In [ ]:
def get_inchi(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            can_smiles = Chem.MolToSmiles(mol, canonical=True)
            return Chem.MolToInchiKey(Chem.MolFromSmiles(can_smiles))
    except: return None
    return None

def deduplicate_and_curate_inchi(df):
    print("Generating InChI keys for robust deduplication...")
    df['inchi_key'] = df['canonical_smiles'].apply(get_inchi)
    df = df.dropna(subset=['inchi_key']).copy()

    stats = df.groupby('inchi_key')['pchembl_value'].agg(['mean', 'min', 'max', 'count']).reset_index()
    # Calculate the spread (Delta) directly in the summary statistics DataFrame
    stats['pchembl_spread'] = stats['max'] - stats['min']
    # Filter out inconsistent molecules (Delta > 1.0) before merging
    stats_valid = stats[stats['pchembl_spread'] <= 1.0].copy()
    # Merge the valid statistics back into the original DataFrame
    df_curated = pd.merge(df.drop_duplicates(subset=['inchi_key']),
                            stats_valid[['inchi_key', 'mean', 'count', 'pchembl_spread']],
                            on='inchi_key')
    # Update the bioactivity label with the consensus mean and remove duplicate rows
    df_curated['pchembl_value'] = df_curated['mean']
    df_curated = df_curated.drop(columns=['mean', 'count', 'pchembl_spread'])
    print(f"Records after deduplication and pChEMBL curation: {len(df_curated)}")
    return df_curated

### **4- Data curated of the bioactivity data**
Combine the 5 columns *(molecule_chembl_id,canonical_smiles,pchembl_value,standard_value, standard_units)* and bioactivity_class into a DataFrame.

In [ ]:
def select_and_save_preprocessed_data(df_curated, target_name):
    selection = ['molecule_chembl_id', 'canonical_smiles', 'pchembl_value', 'standard_value', 'standard_units']
    df_preprocessed = df_curated[selection]
    preprocessed_output_filename = f"{target_name}_bioactivity_data_preprocessed.csv"
    df_preprocessed.to_csv(preprocessed_output_filename, index=False)
    print(f"Preprocessed data saved as: {preprocessed_output_filename}")
    return df_preprocessed

### **5 - Labeling compounds as either being active or inactive**

The bioactivity data is expressed in pChEMBL units. Compounds with values greater than or equal 6 (1 μM) are considered **active**, while those with values less than 6 are categorized as **inactive**.

In [ ]:
def label_and_save_curated_data(df_preprocessed, target_name):
    bioactivity_threshold = []
    for i in df_preprocessed.pchembl_value:
        if float(i) >= 6:
            bioactivity_threshold.append("active")
        elif float(i) < 6:
            bioactivity_threshold.append("inactive")

    bioactivity_class = pd.Series(bioactivity_threshold, name='class')
    df_labeled = pd.concat([df_preprocessed, bioactivity_class], axis=1)

    # Binarize 'class' into 'class_numeric'
    df_labeled['class_numeric'] = df_labeled['class'].map({'inactive': 0, 'active': 1})

    curated_output_filename = f"{target_name}_bioactivity_data_curated.csv"
    df_labeled.to_csv(curated_output_filename, index=False)
    print(f"Curated data saved as: {curated_output_filename}")
    return df_labeled

In [ ]:
def process_bioactivity_target(raw_df, target_name):
    print(f"\n--- Processing {target_name} data ---")
    print(f"Initial number of records: {len(raw_df)}")

    df_step1 = handle_initial_missing_data(raw_df)
    df_step2 = process_pchembl_values(df_step1)
    df_step3 = deduplicate_and_curate_inchi(df_step2)
    df_step4 = select_and_save_preprocessed_data(df_step3, target_name)
    df_final = label_and_save_curated_data(df_step4, target_name)

    return df_final

## **How to process therapeutic targets**

You can follow these steps:

1.  **Upload the raw data CSV for each target**, using the `upload_files()` function. This will give you a new `df` and `uploaded_filename`.
2.  **Extract the target name** from the `uploaded_filename`.
3.  **Call the `process_bioactivity_target` function** with the new `df` and `target_name`.

### **Applying the functions to the current dataset**

In [ ]:
# Upload raw dataset ({target}_bioactivity_raw_data.csv)
df, uploaded_filename = upload_files()
print(df.shape)
df.head()

Saving LRRK2_G2019S_bioactivity_raw_data.csv to LRRK2_G2019S_bioactivity_raw_data.csv
User uploaded file "LRRK2_G2019S_bioactivity_raw_data.csv" with length 5194458 bytes
(3261, 47)


,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value,variant_type
0,NaN,NaN,6180232,[],CHEMBL1772597,Inhibition of purified N-terminal GST-tagged L...,B,Q5S007,G2019S,BAO_0000190,...,Leucine-rich repeat serine/threonine-protein k...,9606,NaN,NaN,IC50,uM,UO_0000065,NaN,13.2,G2019S
1,NaN,NaN,6180234,[],CHEMBL1772599,Inhibition of purified N-terminal GST-tagged L...,B,Q5S007,G2019S,BAO_0000190,...,Leucine-rich repeat serine/threonine-protein k...,9606,NaN,NaN,IC50,uM,UO_0000065,NaN,4.1,G2019S
2,NaN,NaN,10840482,[],CHEMBL2015542,Inhibition of LRRK2 G2019S mutant expressed in...,B,Q5S007,G2019S,BAO_0000190,...,Leucine-rich repeat serine/threonine-protein k...,9606,NaN,NaN,IC50,nM,UO_0000065,NaN,6.0,G2019S
3,NaN,NaN,10840483,[],CHEMBL2015542,Inhibition of LRRK2 G2019S mutant expressed in...,B,Q5S007,G2019S,BAO_0000190,...,Leucine-rich repeat serine/threonine-protein k...,9606,NaN,NaN,IC50,nM,UO_0000065,NaN,6.1,G2019S
4,NaN,NaN,10840484,[],CHEMBL2015543,Inhibition of LRRK2 A2016T mutant expressed in...,B,Q5S007,A2016T,BAO_0000190,...,Leucine-rich repeat serine/threonine-protein k...,9606,NaN,NaN,IC50,nM,UO_0000065,NaN,2450.0,G2019S


In [ ]:
# The 'df' and 'uploaded_filename' variables from the initial upload are still available.
# We need to extract the target name from uploaded_filename correctly.
current_target_name = uploaded_filename.replace('_bioactivity_raw_data.csv', '')

# Call the function that processing bioactivity
df_curated = process_bioactivity_target(df, current_target_name)

print(f'First 5 rows of the curated {current_target_name} data:')
display(df_curated.head())


--- Processing LRRK2_G2019S data ---
Initial number of records: 3261
Records after dropping nulls in 'standard_value'/'canonical_smiles': 3241
Records without pChEMBL: 808 (24.93%)
Records after rescuing and dropping null 'pchembl_value': 2433
Generating InChI keys for robust deduplication...
Records after deduplication and pChEMBL curation: 1684
Preprocessed data saved as: LRRK2_G2019S_bioactivity_data_preprocessed.csv
Curated data saved as: LRRK2_G2019S_bioactivity_data_curated.csv
First 5 rows of the curated LRRK2_G2019S data:


,molecule_chembl_id,canonical_smiles,pchembl_value,standard_value,standard_units,class,class_numeric
0,CHEMBL1771409,Cc1cc(N/N=C/c2ccc(O)c(O)c2)nc2ccccc12,4.88000,13200.0,nM,inactive,0
1,CHEMBL1771411,Cc1cc(N/N=C/c2ccncc2)nc2ccccc12,5.39000,4100.0,nM,inactive,0
2,CHEMBL2204495,O=C(Nc1cccnc1)c1cc(-c2ccnc(F)c2)ccc1OCc1ccccc1,7.58875,61.3,nM,active,1
3,CHEMBL2333124,Cc1c(-c2ccncc2)ccc2c(NC(C)C)c(C(N)=O)nnc12,5.60000,2512.0,nM,inactive,0
4,CHEMBL2333123,CNC(=O)c1nnc2cc(-c3ccc(S(C)(=O)=O)cc3)ccc2c1N[...,6.69000,247.0,nM,active,1


---